# 07 リスク水準を統一した戦略比較

## 1. 研究の目的と仮説
SPY・TLT・GLDの等ウェイト・逆ボラ・最小分散を共通の**予測**年率ボラ10%で比較します。
低い変動率が実際に良い運用成績を意味するか、収益・下方リスク・費用・借入も調べます。
8/10/12%・上限1.5倍等は既存結果を見た後の追加条件です。独立した未見将来の検証ではありません。
実現ボラで事後的に各戦略を同じ値へ合わせる操作はしません。
[設計](../資料/06_リスク統一比較の設計.md) / [コードと成果物の案内](../資料/07_研究コードと成果物の案内.md)。

## 2. 理論・数式と指標
月末tまでのL=252個の資産日次リターンから共分散を求め、元の配分 $b_t$ に対して
$$\hat\sigma_t=\sqrt{252 b_t^\top\hat\Sigma_t b_t},\quad
\ell_t=\frac{\sigma^*}{\hat\sigma_t},\quad w_t=\ell_t b_t.$$
制約付きは $\ell_t=\min(\ell_t,1.5)$。預金比率は $1-\ell$、負なら借入です。
これは月次の目標上限であり、月中の実現倍率はドリフトで超過し得ます。

約定前NAV $V^-$ と資産金額 $H_i^-$ に対し、コスト率cについて
$$V^+=V^- - c\sum_i|w_i V^+-H_i^-|$$
を解いて自己資金制約を満たします。旧保有の当日損益・利息を計算してから翌終値で約定します。
金利は年実効率r、d暦日の利息率 $(1+r)^{d/365}-1$。週末を含めます。

CAGR $=\prod(1+r_{net})^{252/n}-1$。
Sharpe $=\sqrt{252}\,\overline{r_{net}}/s(r_{net})$（基本rf=0）。分子は日次算術平均でCAGRではありません。
SortinoはMAR=0、下方偏差 $\sqrt{\operatorname{mean}(\min(r,0)^2)}$（分母は全日数）。
Calmar=CAGR/|最大DD|、ES95は日次下位ceil(5%n)の平均損失、年率化しません。
最大DDからの回復は底→回復の暦日、未回復は打切り。CAGR・DDを倍率で単純乗算しません。

## 3. データの定義・取得・調整
既存のYahoo/yfinance調整終値スナップショットを再利用します。USD・配当再投資相当、
2004-11-18〜2026-10-29。10月30日は元取得時に欠損のため共通観測から除外済みです。
新規取得・補完・元データ変更はありません。税・JPY換算・当時のデータ版は未対応です。
これは実データ分析で、合成データを使う会計・約定テストとは別です。

## 4. 分析条件
評価2006-01-03〜2026-10-29、月次判断・翌共通取引日終値約定、片道5bps。
全条件はCLIで計算済み。窓比較126/252/504日は全手法2007年開始にそろえます。
金利0/0%、2/3%、4/6%は預金/借入の固定仮想年率で、歴史的金利の再現ではありません。
コストは0/5/20bps。上限なしと月次目標1.5倍を区別します。
全再計算は `python scripts/run_risk_normalized_study.py`（既存出力を上書きしません）。

In [ ]:
from pathlib import Path
import os
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Code
from quant_ai.volatility_target import CORE_METHODS, volatility_schedule
from quant_ai.leveraged_backtest import leveraged_backtest
from quant_ai.downside_metrics import evaluate_leveraged
from quant_ai.risk_research import load_risk_research, conditional_intervals
from quant_ai.research_plots import research_figures

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/quant_ai").exists())
os.chdir(ROOT)
prices = pd.read_parquet(ROOT / "data/processed/spy_tlt_gld_adjusted.parquet")
bundle = load_risk_research(ROOT)
display(pd.Series(bundle.provenance))
TARGET = .10
LOOKBACK = 252
COST_BPS = 5


## 5. 実際のPython実装
[目標生成](../src/quant_ai/volatility_target.py)、
[約定・借入エンジン](../src/quant_ai/leveraged_backtest.py)、
[評価指標](../src/quant_ai/downside_metrics.py)、
[全条件の実行](../src/quant_ai/risk_research.py)。実装を表示し、6ケースをこのNotebookでも再計算します。

In [ ]:
display(Code(inspect.getsource(volatility_schedule), language="python"))
display(Code(inspect.getsource(leveraged_backtest), language="python"))
recomputed = {}
for cap, label in [(None, "unbounded"), (1.5, "cap15")]:
    for method in CORE_METHODS:
        targets, audit = volatility_schedule(prices, method, lookback=LOOKBACK,
                                             target_volatility=TARGET, leverage_cap=cap)
        result = leveraged_backtest(prices.loc[targets.index[0]:], targets, cost_bps=COST_BPS)
        key = f"{label}_{method}"
        np.testing.assert_allclose(result.daily.net_return, bundle.daily[key].daily.net_return,
                                   atol=1e-12)
        recomputed[key] = evaluate_leveraged(result, TARGET, cap)
display(pd.DataFrame(recomputed).T[["annual_return", "volatility", "sharpe", "max_drawdown"]])
display(bundle.audits["cap15_min_variance"].head())


## 6. 結果と可視化
上の再計算と保存結果の一致を確認済み。旧100%投資も含む全指標と9図を確認します。
PNGは`outputs/figures/リスク統一比較/`に保存。同じ関数をCLIとNotebookで共用します。
金利感応度の図だけは預金/借入の条件を変えています。図中のcap15は月次目標上限です。

In [ ]:
primary = bundle.summary[bundle.summary.family.isin(["original", "primary"])]
display(primary[["annual_return", "volatility", "sharpe", "sortino", "max_drawdown", "calmar",
                 "daily_es95_loss", "trough_to_recovery_days", "recovery_censored",
                 "annual_turnover", "total_trade_cost_initial_capital", "mean_held_leverage",
                 "max_realized_leverage", "realized_minus_target_volatility"]])
figures = research_figures(bundle, prices, ROOT / "outputs/figures/リスク統一比較")
for name, figure in figures.items():
    print(name)
    display(figure)
    plt.close(figure)


## 7. 統計的解釈・不確実性
21/63日の循環ブロックをペア再標本化、各2000回。コードと実際の再計算を確認します。
これは**既存戦略リターンに条件付きの区間**で、重みの再推定・非定常性・選択バイアスは十分反映しません。
0をまたぐ差から優位を主張せず、0をまたがなくても正式な一般化された優位性検定とは呼びません。

In [ ]:
display(Code(inspect.getsource(conditional_intervals), language="python"))
returns = pd.DataFrame({k: v.daily.net_return for k,v in bundle.daily.items()
                        if not k.startswith("original")})
pairs = [(f"{cap}_{method}", f"{cap}_equal_weight")
         for cap in ["unbounded", "cap15"] for method in CORE_METHODS[1:]]
intervals = pd.concat([conditional_intervals(returns, pairs, block=b, repetitions=2000)
                       for b in [21, 63]], ignore_index=True)
np.testing.assert_allclose(intervals[["lower_95", "upper_95"]],
                           bundle.bootstrap[["lower_95", "upper_95"]], atol=1e-12)
display(intervals)
display(bundle.summary[bundle.summary.family.isin(["lookback_sensitivity",
                                                  "target_sensitivity", "funding_sensitivity",
                                                  "cost_sensitivity"])][
    ["family", "method", "target_volatility", "leverage_cap", "lookback", "cash_rate",
     "borrow_rate", "cost_bps", "annual_return", "volatility", "max_drawdown"]])


## 8. 金融工学・運用上の解釈
期待リスクを揃えると、低ボラの基本配分にはより大きな投資倍率が割り当てられます。
予測誤差・月中ドリフト・資金調達・複利があるため、実現リスクや下方損失まで揃うとは限りません。
月次目標1.5倍の実現倍率が超過していないか、借入日・CAGR低下・ES・回復日数も確認します。
株債相関の変化は説明候補であり、この分析で因果的な寄与を特定したものではありません。

## 9. 限界・追加検証と投資信託採用判断
現時点で投資信託として採用可とは判断できません。今回の数値は条件付きの研究結果です。
独立の将来検証、別資産・JPY評価、時変金利と実借入契約、解約資金・流動性・インパクト・
税・管理報酬、上限超過時の強制縮小・証拠金、ストレステストと運用/監視体制が不足しています。
無上限は理論例であり、実際の商品規約で実行可能という意味ではありません。
詳しい結果と事実/統計/理論/推測/実務課題の区別は
[日本語分析レポート](../outputs/reports/リスク統一比較_分析レポート.md)に記載しています。